In [1]:
import duckdb
import numpy as np
import pandas as pd

In [2]:
#   Analysing a particular scenario
OBSERVATIONS_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/observations.jsonl"
LABELS_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/labels.jsonl"
SUPPORTING_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/supporting.jsonl"

#   JSON
SNAPSHOT_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/snapshot.json"

In [3]:
def process_sql_query(query: str):
    return duckdb.query(query).df()


def deduplicating_df(df: pd.DataFrame, subset: str) -> pd.DataFrame:
    df = df.drop_duplicates(subset=[subset], keep=False)
    
    return df.reset_index(drop=True)


def process_observations(observations_path: str) -> pd.DataFrame:
    observations_df = pd.read_json(observations_path, lines=True)
    observations_df = deduplicating_df(observations_df, "observation_id")

    #   Removing all rows with invalid latitude and longitude
    observations_df = observations_df[
        observations_df["latitude"].between(-90.0, 90.0) &
        observations_df["longitude"].between(-180.0, 180.0)
    ]

    #   Converting to DateTime for splink
    observations_df['observed_at'] = pd.to_datetime(observations_df['observed_at'])
    observations_df['received_at'] = pd.to_datetime(observations_df['received_at'])

    attributes_df = pd.json_normalize(observations_df["attributes"]).add_prefix("observ_")
    observations_df = pd.concat(
        [observations_df.drop(columns=["attributes"]), attributes_df],
        axis=1
    )


    if 'observ_spd' in observations_df.columns:
        is_feed_a = (observations_df['source'] == 'feed-a') & (observations_df['observ_spd'].notna())
        observations_df.loc[is_feed_a, 'observ_spd'] = observations_df.loc[is_feed_a, 'observ_spd'] * 0.514444

    return observations_df




def process_supporting(supporting_path: str) -> pd.DataFrame:
    supporting_df = pd.read_json(supporting_path, lines=True)
    supporting_df = deduplicating_df(supporting_df, "record_id")

    if 'received_at' in supporting_df.columns:
        supporting_df['received_at'] = pd.to_datetime(supporting_df['received_at'])


    if "attributes" in supporting_df.columns:
        attributes_df_2 = pd.json_normalize(supporting_df["attributes"]).add_prefix("supporting_")
        supporting_df = pd.concat(
            [supporting_df.drop(columns=["attributes"]), attributes_df_2],
            axis=1
        )

    return supporting_df




def read_scenario(observations_path: str, labels_path: str, supporting_path: str, snapshot_path: str):
    observations_df = process_observations(observations_path)    
    labels_df = pd.read_json(labels_path, lines=True)
    supporting_df = process_supporting(supporting_path)
    snapshot_df = pd.read_json(snapshot_path)       

    return observations_df, labels_df, supporting_df, snapshot_df





def table_join(observations_df, supporting_df):
    merged_df = pd.merge(
        observations_df,
        supporting_df,
        on="observation_id",
        how="left",
        suffixes=('', '_supp')
    )

    #   Sorting by tracklet_id and time (Probably the most important step)
    merged_df = merged_df.sort_values(by=["tracklet_id", "observed_at"]).reset_index(drop=True)
    return merged_df



def aggregation(merged_df: pd.DataFrame) -> pd.DataFrame:
    #   The clarifying document mentions:
    #   "Each tracklet contains observations from exactly one aircraft. 
    #   Its membership is fixed. You do not need to split tracklets, merge duplicate tracklets, 
    #   or question whether observations within a tracklet belong together."
    agg_funcs = {
        'observed_at': ['first', 'last'],
        'latitude': ['first', 'last'],
        'longitude': ['first', 'last'],
        
        'source': ['first'],
        'observ_ref': ['first'],
        
        'observ_spd': ['mean'],
        'observ_hdg': ['mean'],
        'supporting_spd': ['mean']
    }

    tracklets_summary = merged_df.groupby('tracklet_id').agg(agg_funcs).reset_index()
    tracklets_summary.columns = [
        'tracklet_id', 
        'start_time', 'end_time', 
        'start_lat', 'end_lat', 
        'start_lon', 'end_lon',
        'source',      
        'observ_ref',  
        'observ_spd',  
        'observ_hdg',  
        'supporting_spd'
    ]

    return tracklets_summary

In [4]:
observations_df, labels_df, supporting_df, snapshot_df = read_scenario(
    OBSERVATIONS_PATH,
    LABELS_PATH,
    SUPPORTING_PATH,
    SNAPSHOT_PATH
)

print(observations_df.shape)
print(labels_df.shape)
print(supporting_df.shape)
print(snapshot_df.shape)

(302, 18)
(24, 2)
(180, 8)
(3, 10)


In [5]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates the great-circle distance between two points in meters."""

    lat1 = np.asarray(lat1, dtype=float)
    lon1 = np.asarray(lon1, dtype=float)
    lat2 = np.asarray(lat2, dtype=float)
    lon2 = np.asarray(lon2, dtype=float)
    
    R = 6371000.0 
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi/2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2.0)**2
    return 2.0 * R * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))



def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1 = np.asarray(lat1, dtype=float)
    lon1 = np.asarray(lon1, dtype=float)
    lat2 = np.asarray(lat2, dtype=float)
    lon2 = np.asarray(lon2, dtype=float)
    
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    lambda1, lambda2 = np.radians(lon1), np.radians(lon2)
    dlambda = lambda2 - lambda1
    
    y = np.sin(dlambda) * np.cos(phi2)
    x = np.cos(phi1) * np.sin(phi2) - np.sin(phi1) * np.cos(phi2) * np.cos(dlambda)
    
    bearing = np.degrees(np.arctan2(y, x))
    return (bearing + 360.0) % 360.0
    

In [6]:
def prepare_features_and_labels(tracklets_summary: pd.DataFrame, labels_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates all candidate pairs (A -> B), computes physical relationship features,
    and attaches binary targets based on the labels.
    """
    
    df_a = tracklets_summary.add_prefix('A_')
    df_b = tracklets_summary.add_prefix('B_')

    pairs = pd.merge(df_a.assign(key=1), df_b.assign(key=1), on='key').drop('key', axis=1)
    pairs = pairs[pairs['A_tracklet_id'] != pairs['B_tracklet_id']]
    
    pairs = pairs[pairs['B_start_time'] > pairs['A_end_time']].copy()
    pairs['gap_time_seconds'] = (pairs['B_start_time'] - pairs['A_end_time']).dt.total_seconds()
    
    pairs['gap_distance_meters'] = haversine_distance(
        pairs['A_end_lat'], pairs['A_end_lon'], 
        pairs['B_start_lat'], pairs['B_start_lon']
    )

    pairs['required_speed_mps'] = pairs['gap_distance_meters'] / (pairs['gap_time_seconds'] + 1e-6)
    pairs['gap_bearing'] = calculate_bearing(
        pairs['A_end_lat'], pairs['A_end_lon'], 
        pairs['B_start_lat'], pairs['B_start_lon']
    )

    if 'A_observ_ref' in pairs.columns:
        pairs['ref_match_exact'] = (
            (pairs['A_observ_ref'] == pairs['B_observ_ref']) & 
            (pairs['A_observ_ref'].notna())
        ).astype(int)

    if 'A_source' in pairs.columns:
        pairs['is_same_source'] = (pairs['A_source'] == pairs['B_source']).astype(int)

    if 'A_observ_spd' in pairs.columns:
        pairs['speed_diff_A_observ'] = abs(pairs['required_speed_mps'] - pairs['A_observ_spd'])
        pairs['speed_diff_B_observ'] = abs(pairs['required_speed_mps'] - pairs['B_observ_spd'])

    if 'A_supporting_spd' in pairs.columns:
        pairs['telemetry_speed_diff'] = abs(pairs['A_supporting_spd'] - pairs['B_supporting_spd'])


    # Attach the Ground Truth Labels (The Target Variable)
    # Rename labels to match pair structure for a clean left join
    truth_df = labels_df[['tracklet_id', 'successor_tracklet_id']].copy()
    truth_df = truth_df.rename(columns={
        'tracklet_id': 'A_tracklet_id', 
        'successor_tracklet_id': 'B_tracklet_id'
    })
    
    # Add a column so we know which pairs matched the truth
    truth_df['target'] = 1
    
    # Left join the truth onto our pairs
    final_dataset = pd.merge(
        pairs, 
        truth_df, 
        on=['A_tracklet_id', 'B_tracklet_id'], 
        how='left'
    )
    
    # Any pair that wasn't in the truth table gets a 0 (False link)
    final_dataset['target'] = final_dataset['target'].fillna(0).astype(int)
    
    # Clean up the dataframe for the ML model
    # Drop timestamps as ML models can't ingest raw datetimes directly
    cols_to_drop = ['A_start_time', 'A_end_time', 'B_start_time', 'B_end_time']
    final_dataset = final_dataset.drop(columns=cols_to_drop, errors='ignore')
    
    return final_dataset

In [7]:
master_df = table_join(observations_df, supporting_df)
aggregated_df = aggregation(master_df)
ml_dataset = prepare_features_and_labels(aggregated_df, labels_df)

print(master_df.shape)
print(aggregated_df.shape)
print(ml_dataset.shape)

(302, 25)
(24, 12)
(216, 30)


In [8]:
X = ml_dataset.drop(columns=['target', 'A_tracklet_id', 'B_tracklet_id'])
X = X.select_dtypes(exclude=['object', 'string'])

y = ml_dataset['target']

In [9]:
X.shape

(216, 23)

In [10]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   A_start_lat           216 non-null    float64
 1   A_end_lat             216 non-null    float64
 2   A_start_lon           216 non-null    float64
 3   A_end_lon             216 non-null    float64
 4   A_observ_spd          138 non-null    float64
 5   A_observ_hdg          78 non-null     float64
 6   A_supporting_spd      216 non-null    float64
 7   B_start_lat           216 non-null    float64
 8   B_end_lat             216 non-null    float64
 9   B_start_lon           216 non-null    float64
 10  B_end_lon             216 non-null    float64
 11  B_observ_spd          150 non-null    float64
 12  B_observ_hdg          66 non-null     float64
 13  B_supporting_spd      216 non-null    float64
 14  gap_time_seconds      216 non-null    float64
 15  gap_distance_meters   216 non-null

In [11]:
X.head()

,A_start_lat,A_end_lat,A_start_lon,A_end_lon,A_observ_spd,A_observ_hdg,A_supporting_spd,B_start_lat,B_end_lat,B_start_lon,...,B_supporting_spd,gap_time_seconds,gap_distance_meters,required_speed_mps,gap_bearing,ref_match_exact,is_same_source,speed_diff_A_observ,speed_diff_B_observ,telemetry_speed_diff
0,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.313578,35.238779,-108.546265,...,49.000000,553.735,95398.564599,172.281984,250.694544,0,1,9.022723,1.124984,4.000000
1,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.378941,35.315701,-107.897465,...,37.750000,324.964,39728.696485,122.255685,231.614246,0,1,59.049023,42.604969,15.250000
2,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.300919,35.206605,-107.972739,...,40.500000,319.779,50540.482303,158.048159,228.757163,0,0,23.256549,3.565687,12.500000
3,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.041899,34.959947,-108.285103,...,42.875000,558.109,90933.325168,162.931121,227.052662,0,0,18.373586,49.123879,10.125000
4,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.378185,35.290872,-107.871079,...,52.916667,318.097,37943.051592,119.281387,229.261724,0,1,62.023320,27.232264,0.083333


In [12]:
y.shape

(216,)

In [13]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64

In [14]:
#   There are only 18 rows marked as 1. All others are 0
y.sum().item()

18

### Processing all scenarios together

In [15]:
def build_all_features(observations_path: str, labels_path: str, supporting_path: str, snapshot_path: str):
    observations_df, labels_df, supporting_df, snapshot_df = read_scenario(
        observations_path,
        labels_path,
        supporting_path,
        snapshot_path
    )
    
    master_df = table_join(observations_df, supporting_df)
    aggregated_df = aggregation(master_df)
    ml_dataset = prepare_features_and_labels(aggregated_df, labels_df)

    return ml_dataset


In [26]:
def build_complete_dataset(base_path: str, limit: int):
    all_scenarios_data = []

    for i in range(1, limit+1):
        OBSERVATIONS_PATH = f"{base_path}/scenario_{i:03d}/observations.jsonl"
        LABELS_PATH = f"{base_path}/scenario_{i:03d}/labels.jsonl"
        SUPPORTING_PATH = f"{base_path}/scenario_{i:03d}/supporting.jsonl"
        SNAPSHOT_PATH = f"{base_path}/scenario_{i:03d}/snapshot.json"

        print(f"Processing scenario {i:03d}...")

        scenario_df = build_all_features(OBSERVATIONS_PATH, LABELS_PATH, SUPPORTING_PATH, SNAPSHOT_PATH)
        all_scenarios_data.append(scenario_df)

        print()


    full_dataset = pd.concat(all_scenarios_data, ignore_index=True)
    X = full_dataset.drop(columns=['target', 'A_tracklet_id', 'B_tracklet_id'], errors='ignore')
    X = X.select_dtypes(exclude=['object', 'string'])
    
    y = full_dataset['target']

    return X, y

In [27]:
train_path = f"/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/"
test_path = f"/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/test/"
val_path = f"/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/validation/"

X_train, y_train = build_complete_dataset(train_path, 80)
X_test, y_test = build_complete_dataset(test_path, 20)
X_val, y_val = build_complete_dataset(val_path, 20)

Processing scenario 001...

Processing scenario 002...

Processing scenario 003...

Processing scenario 004...

Processing scenario 005...

Processing scenario 006...

Processing scenario 007...

Processing scenario 008...

Processing scenario 009...

Processing scenario 010...

Processing scenario 011...

Processing scenario 012...

Processing scenario 013...

Processing scenario 014...

Processing scenario 015...

Processing scenario 016...

Processing scenario 017...

Processing scenario 018...

Processing scenario 019...

Processing scenario 020...

Processing scenario 021...

Processing scenario 022...

Processing scenario 023...

Processing scenario 024...

Processing scenario 025...

Processing scenario 026...

Processing scenario 027...

Processing scenario 028...

Processing scenario 029...

Processing scenario 030...

Processing scenario 031...

Processing scenario 032...

Processing scenario 033...

Processing scenario 034...

Processing scenario 035...

Processing scenario 

In [28]:
print(X_train.shape)
print(X_test.shape)
print(X_val.shape)

(15912, 23)
(3942, 23)
(3870, 23)


In [29]:
X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 15912 entries, 0 to 15911
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   A_start_lat           15912 non-null  float64
 1   A_end_lat             15912 non-null  float64
 2   A_start_lon           15912 non-null  float64
 3   A_end_lon             15912 non-null  float64
 4   A_observ_spd          10821 non-null  float64
 5   A_observ_hdg          5091 non-null   float64
 6   A_supporting_spd      15895 non-null  float64
 7   B_start_lat           15912 non-null  float64
 8   B_end_lat             15912 non-null  float64
 9   B_start_lon           15912 non-null  float64
 10  B_end_lon             15912 non-null  float64
 11  B_observ_spd          10427 non-null  float64
 12  B_observ_hdg          5485 non-null   float64
 13  B_supporting_spd      15912 non-null  float64
 14  gap_time_seconds      15912 non-null  float64
 15  gap_distance_meters   15912 no

In [30]:
X_train.head()

,A_start_lat,A_end_lat,A_start_lon,A_end_lon,A_observ_spd,A_observ_hdg,A_supporting_spd,B_start_lat,B_end_lat,B_start_lon,...,B_supporting_spd,gap_time_seconds,gap_distance_meters,required_speed_mps,gap_bearing,ref_match_exact,is_same_source,speed_diff_A_observ,speed_diff_B_observ,telemetry_speed_diff
0,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.313578,35.238779,-108.546265,...,49.000000,553.735,95398.564599,172.281984,250.694544,0,1,9.022723,1.124984,4.000000
1,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.378941,35.315701,-107.897465,...,37.750000,324.964,39728.696485,122.255685,231.614246,0,1,59.049023,42.604969,15.250000
2,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.300919,35.206605,-107.972739,...,40.500000,319.779,50540.482303,158.048159,228.757163,0,0,23.256549,3.565687,12.500000
3,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.041899,34.959947,-108.285103,...,42.875000,558.109,90933.325168,162.931121,227.052662,0,0,18.373586,49.123879,10.125000
4,35.671734,35.601289,-107.336176,-107.553976,181.304707,NaN,53.0,35.378185,35.290872,-107.871079,...,52.916667,318.097,37943.051592,119.281387,229.261724,0,1,62.023320,27.232264,0.083333


In [31]:
y_train.shape

(15912,)

In [32]:
print(y_train.sum().item())
print(y_test.sum().item())
print(y_val.sum().item())

1364
339
335


### Model fitting and evaluation:

In [33]:
#   Since incorrectly joining fragments is worse than leaving them unresolved (as stated in the challenge rules), we want to focus heavily on Precision

import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [34]:
pos_weight = (y == 0).sum() / (y == 1).sum()

model = xgb.XGBClassifier(
    scale_pos_weight=pos_weight,
    eval_metric='aucpr',  
    max_depth=4,          
    learning_rate=0.1,
    n_estimators=150,
    random_state=42
)

In [35]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [36]:

# 5. Evaluate the model
y_pred_proba = model.predict_proba(X_val)[:, 1]

# Set a strict threshold. 
# The rules state: "Leaving a case unresolved is acceptable. Incorrectly joining different aircraft is more costly."
# So we require 70% confidence to predict a link, rather than the default 50%.
strict_threshold = 0.70
y_pred_strict = (y_pred_proba >= strict_threshold).astype(int)

print("\n--- Confusion Matrix (Threshold = 0.70) ---")
print(confusion_matrix(y_val, y_pred_strict))

print("\n--- Classification Report ---")
print(classification_report(y_val, y_pred_strict))

# 6. Feature Importance (To see which physics rules XGBoost loved the most)
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n--- Top 5 Most Important Features ---")
print(importance.head(5))


--- Confusion Matrix (Threshold = 0.70) ---
[[3263  272]
 [  75  260]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      0.92      0.95      3535
           1       0.49      0.78      0.60       335

    accuracy                           0.91      3870
   macro avg       0.73      0.85      0.77      3870
weighted avg       0.94      0.91      0.92      3870


--- Top 5 Most Important Features ---
                Feature  Importance
15  gap_distance_meters    0.330688
16   required_speed_mps    0.168791
18      ref_match_exact    0.076376
20  speed_diff_A_observ    0.057005
21  speed_diff_B_observ    0.045017
